In [ ]:
import csv
import logging
import time
from pathlib import Path
from urllib.parse import urlparse

import undetected_chromedriver as uc
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

BASE_DIR = Path('/home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/BaanFinder')
START_URL = "https://www.baanfinder.com/en/properties?ref=see-more"
OUTPUT_CSV_FILE = BASE_DIR / 'baanfinder_listing_urls.csv'

TARGET_PAGES = 5
PAGE_TIMEOUT = 15

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def setup_optimized_driver() -> uc.Chrome:
    options = uc.ChromeOptions()
    options.page_load_strategy = 'eager'

    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.managed_default_content_settings.stylesheets": 2,
        "profile.default_content_setting_values.notifications": 2,
        "profile.managed_default_content_settings.fonts": 2,
    }
    options.add_experimental_option("prefs", prefs)

    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-extensions")

    return uc.Chrome(options=options, version_main=145)

def extract_links(driver: uc.Chrome, base_url: str) -> list[str]:
    js = """
    return [...new Set(
        Array.from(document.querySelectorAll('a.stretched-link'))
        .map(a => a.getAttribute('href'))
        .filter(Boolean)
    )];
    """
    hrefs = driver.execute_script(js)
    return [f"{base_url}{h}" if h.startswith("/") else h for h in hrefs]

def main():
    BASE_DIR.mkdir(parents=True, exist_ok=True)

    driver = setup_optimized_driver()
    wait = WebDriverWait(driver, PAGE_TIMEOUT)
    parsed_url = urlparse(START_URL)
    base_url = f"{parsed_url.scheme}://{parsed_url.netloc}"

    all_urls = set()

    try:
        logging.info(f"Starting scrape at: {START_URL}")
        driver.get(START_URL)
        
        current_page = 1
        
        while current_page <= TARGET_PAGES:
            try:
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'a.stretched-link')))
            except TimeoutException:
                logging.warning(f"Timeout waiting for elements on page {current_page}. Stopping.")
                break
                
            current_links = extract_links(driver, base_url)
            all_urls.update(current_links)
            
            logging.info(f"Page {current_page}: Collected {len(current_links)} URLs. Total: {len(all_urls)}")

            if current_page >= TARGET_PAGES:
                break

            try:
                next_button = driver.find_element(By.ID, 'next')
                next_url = next_button.get_attribute("href")
                driver.get(next_url)
                current_page += 1
                time.sleep(2)
            except NoSuchElementException:
                logging.info("Next page button not found. Reached the end.")
                break

    except KeyboardInterrupt:
        logging.info("Scraping manually interrupted by user.")
    except Exception as e:
        logging.error(f"Unexpected error: {e}")
    finally:
        driver.quit()

    final_urls = sorted(list(all_urls))

    with OUTPUT_CSV_FILE.open('w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['ListingURL'])
        for u in final_urls:
            writer.writerow([u])

    logging.info(f"Successfully saved {len(final_urls)} URLs to {OUTPUT_CSV_FILE}")

if __name__ == "__main__":
    main()

2026-03-29 16:55:28,426 - INFO - patching driver executable /home/kongla/.local/share/undetected_chromedriver/undetected_chromedriver
2026-03-29 16:55:30,942 - INFO - Starting scrape at: https://www.baanfinder.com/en/properties?ref=see-more
2026-03-29 16:55:33,183 - INFO - Page 1: Collected 20 URLs. Total: 20
2026-03-29 16:55:36,229 - INFO - Page 2: Collected 20 URLs. Total: 33
2026-03-29 16:55:39,754 - INFO - Page 3: Collected 20 URLs. Total: 47
2026-03-29 16:55:42,672 - INFO - Page 4: Collected 20 URLs. Total: 67
2026-03-29 16:55:45,422 - INFO - Page 5: Collected 20 URLs. Total: 87
2026-03-29 16:55:45,427 - INFO - Successfully saved 87 URLs to /home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/baanfinder/baanfinder_listing_urls.csv


In [2]:
import csv
import logging
import time
from pathlib import Path

import undetected_chromedriver as uc
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

BASE_DIR = Path('/home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/BaanFinder')
INPUT_CSV = BASE_DIR / 'baanfinder_listing_urls.csv'
OUTPUT_CSV = BASE_DIR / 'baanfinder_full_details.csv'

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def setup_driver() -> uc.Chrome:
    options = uc.ChromeOptions()
    options.page_load_strategy = 'eager'
    
    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.default_content_setting_values.notifications": 2,
        "profile.managed_default_content_settings.stylesheets": 2,
        "profile.managed_default_content_settings.fonts": 2,
    }
    options.add_experimental_option("prefs", prefs)
    
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-notifications")
    options.add_argument("--disable-extensions")
    
    return uc.Chrome(options=options, version_main=145)

def extract_content(driver: uc.Chrome) -> str:
    data = []
    
    try:
        title = driver.find_element(By.CSS_SELECTOR, '.res-title h1').text.strip()
        data.extend(["--- Listing Title ---", title])
    except NoSuchElementException:
        pass

    try:
        price_elem = driver.find_element(By.CSS_SELECTOR, '.post-type .price')
        price = price_elem.text.replace('\n', ' ').strip()
        data.extend(["\n--- Price ---", price])
    except NoSuchElementException:
        pass

    try:
        quick_summary = driver.find_element(By.CSS_SELECTOR, '.quick-summary')
        
        try:
            prop_type = quick_summary.find_element(By.CSS_SELECTOR, '.type-n-group a').text.strip()
            data.extend(["\n--- Property Type ---", prop_type])
        except NoSuchElementException:
            pass

        try:
            area_elements = quick_summary.find_elements(By.CSS_SELECTOR, '.area-amount')
            if area_elements:
                area = " ".join([elem.text.strip() for elem in area_elements])
                data.extend(["\n--- Area Size ---", area])
        except NoSuchElementException:
            pass
            
        try:
            price_per_area = quick_summary.find_element(By.CSS_SELECTOR, '.pricePerArea').text.replace('\n', ' ').strip()
            data.extend(["\n--- Price Per Area ---", price_per_area])
        except NoSuchElementException:
            pass

        try:
            location = quick_summary.find_element(By.CSS_SELECTOR, '.province').text.replace('\n', ' ').strip()
            data.extend(["\n--- Location ---", location])
        except NoSuchElementException:
            pass

    except NoSuchElementException:
        pass

    try:
        details_header = driver.find_element(By.XPATH, "//h4[contains(text(), 'Details')]")
        detailed_info_div = details_header.find_element(By.XPATH, "following-sibling::div[contains(@class, 'detailedInfo')]")
        
        try:
            read_more_btn = detailed_info_div.find_element(By.ID, 'js-readMore')
            driver.execute_script("arguments[0].click();", read_more_btn)
            time.sleep(0.5) 
        except NoSuchElementException:
            pass
            
        desc = detailed_info_div.find_element(By.ID, 'js-detailed-info').text.strip()
        if desc:
            data.extend(["\n--- Description ---", desc])
            
    except NoSuchElementException:
        pass
        
    try:
        last_updated = driver.find_element(By.CSS_SELECTOR, '.res-date').text.strip()
        if last_updated:
            data.extend(["\n--- Updated Info ---", last_updated])
    except NoSuchElementException:
        pass

    return "\n".join(data)

def main():
    BASE_DIR.mkdir(parents=True, exist_ok=True)

    if not INPUT_CSV.exists():
        logging.warning(f"File not found: {INPUT_CSV}")
        return

    with INPUT_CSV.open('r', encoding='utf-8') as f:
        reader = csv.reader(f)
        next(reader, None)
        urls = [row[0] for row in reader if row]

    if not urls:
        logging.info("No URLs found to scrape.")
        return

    driver = setup_driver()
    wait = WebDriverWait(driver, 10)

    try:
        with OUTPUT_CSV.open('w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['Post_URL', 'Full_Content'])

            for i, url in enumerate(urls, 1):
                try:
                    driver.get(url)
                    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '.res-title h1')))
                    
                    content = extract_content(driver)
                    writer.writerow([url, content])
                    
                    if i % 10 == 0:
                        logging.info(f"Progress: [{i}/{len(urls)}] scraped.")

                except TimeoutException:
                    logging.error(f"[{i}/{len(urls)}] Timeout loading: {url}")
                except Exception as e:
                    logging.error(f"[{i}/{len(urls)}] Error scraping {url}: {e}")

    finally:
        driver.quit()
        logging.info(f"Scraping completed. Saved to: {OUTPUT_CSV}")

if __name__ == "__main__":
    main()

2026-03-29 16:57:04,370 - INFO - patching driver executable /home/kongla/.local/share/undetected_chromedriver/undetected_chromedriver
2026-03-29 16:57:31,461 - INFO - Progress: [10/87] scraped.
2026-03-29 16:57:51,134 - INFO - Progress: [20/87] scraped.
2026-03-29 16:58:13,059 - INFO - Progress: [30/87] scraped.
2026-03-29 16:58:39,684 - INFO - Progress: [40/87] scraped.
2026-03-29 16:59:01,155 - INFO - Progress: [50/87] scraped.
2026-03-29 16:59:24,219 - INFO - Progress: [60/87] scraped.
2026-03-29 16:59:47,441 - INFO - Progress: [70/87] scraped.
2026-03-29 17:00:11,428 - INFO - Progress: [80/87] scraped.
2026-03-29 17:00:26,578 - INFO - Scraping completed. Saved to: /home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/BaanFinder/baanfinder_full_details.csv
